# DNA-Net для кейсу 4.5: навчання на Kaggle, на тих самих даних, що й YOLO

**Перед запуском:** Settings → Accelerator **GPU T4 x2** (використовується 1 GPU), Internet **On**. У клітинці ДАНІ вже стоять посилання: сирий Anti-UAV410 і пакет команди AntiUAV410_SKY (маніфест відібраних кадрів неба). Додається синтетичне сонце.

**Запуск:** Save Version → Save & Run All. Результат у Output: `log/…/DNANet_*.pth.tar`, `metrics.json`.

Маски DNA-Net робляться з тих самих рамок YOLO: пікселі в рамці, що відрізняються від фону (Otsu), інакше еліпс.

In [ ]:
# ======= ПАРАМЕТРИ =======
EPOCHS = 25        # DNA-Net важка: ~20-30 хв на епоху на T4 при ~20 тис. кадрів. 25 епох укладаються в ліміт
BATCH = 16
PATCH = 256
HOURS = 10.5       # страховка від ліміту 12 год: навчання зупиниться, лишиться останній чекпойнт
VAL_EVERY = 4      # val ще раз проріджуємо (разом з ДАНИМИ це кожен 16-й кадр), щоб тест під час навчання не з'їдав час


In [ ]:
!nvidia-smi -L
%cd /kaggle/working
!rm -rf BasicIRSTD && git clone -q https://github.com/XinyiYing/BasicIRSTD.git

In [ ]:
import os; os.makedirs(os.path.dirname('/kaggle/working/BasicIRSTD/patch_basicirstd.py'), exist_ok=True)
open('/kaggle/working/BasicIRSTD/patch_basicirstd.py', 'w').write(r'''"""Patches for BasicIRSTD (github.com/XinyiYing/BasicIRSTD), run once from its folder.

1. net.py: drop a broken import from skimage test files (crashes on new skimage).
2. train.py:
   * checkpoint every --save_every epochs (default 10, original 50) so Colab disconnects lose little;
   * LR drops at 60% and 85% of --nEpochs (original: fixed epochs 200/300, never reached at 100 epochs);
   * --lr sets the Adam learning rate (original hard-codes 5e-4), needed for fine-tuning;
   * --pretrained loads weights from any path (original checked the name and then loaded the wrong
     variable), used to fine-tune from our own best checkpoint on a new dataset;
   * torch.load(..., weights_only=False) for newer PyTorch.
"""
t = open('train.py').read()
rep = [
    ("opt.scheduler_settings = {'epochs':400, 'step': [200, 300], 'gamma': 0.1}",
     "opt.scheduler_settings = {'epochs':400, 'step': [max(1, int(opt.nEpochs*0.6)-epoch_state), max(2, int(opt.nEpochs*0.85)-epoch_state)], 'gamma': 0.1}"),
    ("opt.optimizer_settings = {'lr': 5e-4}", "opt.optimizer_settings = {'lr': opt.lr}"),
    ("(idx_epoch + 1) % 50 == 0", "(idx_epoch + 1) % opt.save_every == 0"),
    ("(idx_epoch + 1) % 50 != 0", "(idx_epoch + 1) % opt.save_every != 0"),
    ("""        for pretrained_pth in opt.pretrained:
            if opt.dataset_name in pretrained_pth and opt.model_name in pretrained_pth:
                ckpt = torch.load(resume_pth)
                net.load_state_dict(ckpt['state_dict'])""",
     """        for pretrained_pth in opt.pretrained:
            ckpt = torch.load(pretrained_pth, weights_only=False)
            net.load_state_dict(ckpt['state_dict'])
            print('loaded pretrained weights from', pretrained_pth)"""),
    ("torch.load(resume_pth)", "torch.load(resume_pth, weights_only=False)"),
    ("torch.load(save_pth)", "torch.load(save_pth, weights_only=False)"),
    ('parser.add_argument("--seed"',
     'parser.add_argument("--save_every", type=int, default=10)\n'
     'parser.add_argument("--lr", type=float, default=5e-4)\n'
     'parser.add_argument("--seed"'),
]
for a, b in rep:
    assert a in t, 'pattern not found: ' + a[:60]
    t = t.replace(a, b)
open('train.py', 'w').write(t)
n = open('net.py').read().replace('from skimage.feature.tests.test_orb import img\n', '')
open('net.py', 'w').write(n)
print('BasicIRSTD patched')
'''); print('записано', '/kaggle/working/BasicIRSTD/patch_basicirstd.py')

In [ ]:
import os; os.makedirs(os.path.dirname('/kaggle/working/BasicIRSTD/eval_thermal.py'), exist_ok=True)
open('/kaggle/working/BasicIRSTD/eval_thermal.py', 'w').write(r'''"""Case 4.5 metrics for a BasicIRSTD segmentation model.

For every test subset it reports what the TZ asks for, not pixel IoU:
  * Recall over GT targets 5..50 px (and over all GT targets),
  * FP per frame (mean and 95th percentile),
  * FPS of the whole pipeline on 640x512 (normalise -> model -> threshold -> post-filters).

A prediction = connected component of (prob > thr). It is a TP when its centroid falls into a GT
target box expanded by `tol` px. Predictions that hit only GT targets outside 5..50 px are ignored
(neither TP nor FP), as agreed in CLAUDE.md.

Post-filters (all optional): sun/saturation mask, component size limits, top-K by peak probability.

Run from the BasicIRSTD folder:
  python eval_thermal.py --model DNANet --ckpt log/THERMAL-MIX/DNANet_100.pth.tar \
      --subsets THERMAL-MIX-FT-hit THERMAL-MIX-FT-flir THERMAL-MIX-FT-sirst THERMAL-MIX-FT-sirst_background THERMAL-MIX-FT-sun \
      --mean 100.0 --std 30.0 --thr 0.5 --topk 5 --sun --json results.json
Add --sweep to try several thresholds / top-K on one subset list (use the validation set: THERMAL-MIX).
"""
import os, sys, time, json, argparse
import numpy as np, cv2, torch

sys.path.insert(0, os.getcwd())


def build(model_name, ckpt, device):
    from net import Net
    net = Net(model_name=model_name, mode="test")
    sd = torch.load(ckpt, map_location="cpu")
    net.load_state_dict(sd["state_dict"] if "state_dict" in sd else sd)
    return net.to(device).eval()


def gt_targets(mask):
    n, lab, st, _ = cv2.connectedComponentsWithStats((mask > 127).astype(np.uint8), 8)
    out = []
    for i in range(1, n):
        x, y, w, h, a = st[i]
        out.append(dict(x=x, y=y, w=w, h=h, in_scope=bool(min(w, h) >= 5 and max(w, h) <= 50)))
    return out


def sun_mask(img, level=250, min_area=200, grow=31):
    sat = (img >= level).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(sat, 8)
    m = np.zeros_like(sat)
    for i in range(1, n):
        if st[i, 4] >= min_area:
            m[lab == i] = 1
    if m.any():
        m = cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (grow, grow)))
    return m


def detections(prob, img, thr, use_sun, min_side, max_side, topk):
    bw = (prob > thr).astype(np.uint8)
    n, lab, st, cen = cv2.connectedComponentsWithStats(bw, 8)
    sm = sun_mask(img) if use_sun else None
    dets = []
    for i in range(1, n):
        x, y, w, h, a = st[i]
        cx, cy = cen[i]
        if sm is not None and sm[int(cy), int(cx)]:
            continue
        if max(w, h) < min_side or max(w, h) > max_side:
            continue
        dets.append(dict(cx=float(cx), cy=float(cy), x=int(x), y=int(y), w=int(w), h=int(h),
                         score=float(prob[lab == i].max())))
    dets.sort(key=lambda d: -d["score"])
    return dets[:topk] if topk > 0 else dets


def match(dets, gts, tol):
    tp = set(); fp = 0; ign = 0
    for d in dets:
        hit = [j for j, g in enumerate(gts) if g["x"] - tol <= d["cx"] <= g["x"] + g["w"] + tol and g["y"] - tol <= d["cy"] <= g["y"] + g["h"] + tol]
        if not hit:
            fp += 1
        elif any(gts[j]["in_scope"] for j in hit):
            tp.update(j for j in hit if gts[j]["in_scope"])
        else:
            ign += 1
    hit_all = set()
    for d in dets:
        hit_all.update(j for j, g in enumerate(gts) if g["x"] - tol <= d["cx"] <= g["x"] + g["w"] + tol and g["y"] - tol <= d["cy"] <= g["y"] + g["h"] + tol)
    return len(tp), fp, len(hit_all)


def pad32(a):
    h, w = a.shape
    H, W = (h + 31) // 32 * 32, (w + 31) // 32 * 32
    return np.pad(a, ((0, H - h), (0, W - w))), (h, w)


@torch.no_grad()
def predict(net, img, mean, std, device, half):
    x, (h, w) = pad32((img.astype(np.float32) - mean) / std)
    t = torch.from_numpy(x)[None, None].to(device)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=half and device == "cuda"):
        p = net(t)
    if isinstance(p, (list, tuple)):
        p = p[-1]
    return p[0, 0, :h, :w].float().cpu().numpy()


def run_subset(net, ds_dir, name, a, device, cache=None):
    ids = open(f"{ds_dir}/{name}/img_idx/test_{name}.txt").read().split()
    tp = n_in = fp_tot = hit_all = n_all = 0
    fps_per_frame = []
    for iid in ids:
        img = cv2.imread(f"{ds_dir}/{name}/images/{iid}.png", 0)
        gts = gt_targets(cv2.imread(f"{ds_dir}/{name}/masks/{iid}.png", 0))
        key = (name, iid)
        if cache is not None and key in cache:
            prob = cache[key]
        else:
            prob = predict(net, img, a.mean, a.std, device, a.half)
            if cache is not None:
                cache[key] = prob
        dets = detections(prob, img, a.thr, a.sun, a.min_side, a.max_side, a.topk)
        t, f, h = match(dets, gts, a.tol)
        tp += t; fp_tot += f; hit_all += h
        n_in += sum(g["in_scope"] for g in gts); n_all += len(gts)
        fps_per_frame.append(f)
    return dict(frames=len(ids), targets_5_50=n_in,
                recall_5_50=round(tp / n_in, 4) if n_in else None,
                recall_all=round(hit_all / n_all, 4) if n_all else None,
                fp_per_frame=round(fp_tot / max(1, len(ids)), 3),
                fp_p95=float(np.percentile(fps_per_frame, 95)) if fps_per_frame else 0.0)


@torch.no_grad()
def bench_fps(net, a, device, n=200):
    img = (np.random.rand(512, 640) * 255).astype(np.uint8)
    for _ in range(10):
        detections(predict(net, img, a.mean, a.std, device, a.half), img, a.thr, a.sun, a.min_side, a.max_side, a.topk)
    if device == "cuda":
        torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(n):
        prob = predict(net, img, a.mean, a.std, device, a.half)
        detections(prob, img, a.thr, a.sun, a.min_side, a.max_side, a.topk)
    if device == "cuda":
        torch.cuda.synchronize()
    ms = (time.perf_counter() - t) / n * 1000
    return dict(ms_per_frame=round(ms, 2), fps=round(1000 / ms, 1), device=torch.cuda.get_device_name(0) if device == "cuda" else "cpu", fp16=a.half)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="DNANet")
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--dataset_dir", default="./datasets")
    ap.add_argument("--subsets", nargs="+", required=True)
    ap.add_argument("--mean", type=float, required=True)
    ap.add_argument("--std", type=float, required=True)
    ap.add_argument("--thr", type=float, default=0.5)
    ap.add_argument("--tol", type=int, default=3)
    ap.add_argument("--sun", action="store_true")
    ap.add_argument("--min_side", type=int, default=2)
    ap.add_argument("--max_side", type=int, default=80)
    ap.add_argument("--topk", type=int, default=0, help="0 = keep all")
    ap.add_argument("--half", action="store_true")
    ap.add_argument("--sweep", action="store_true")
    ap.add_argument("--fps", action="store_true")
    ap.add_argument("--json", default=None)
    a = ap.parse_args()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    net = build(a.model, a.ckpt, device)
    out = {"model": a.model, "ckpt": a.ckpt, "settings": {k: getattr(a, k) for k in ["thr", "tol", "sun", "min_side", "max_side", "topk", "half"]}}

    if a.sweep:
        cache, rows = {}, []
        for thr in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
            for topk in [0, 5, 3]:
                a.thr, a.topk = thr, topk
                r = {s: run_subset(net, a.dataset_dir, s, a, device, cache) for s in a.subsets}
                worst_fp = max(v["fp_per_frame"] for v in r.values())
                rec = np.mean([v["recall_5_50"] for v in r.values() if v["recall_5_50"] is not None])
                rows.append(dict(thr=thr, topk=topk, mean_recall_5_50=round(float(rec), 4), worst_fp_per_frame=worst_fp))
                print(rows[-1], flush=True)
        ok = [r for r in rows if r["worst_fp_per_frame"] <= 0.7]
        out["sweep"] = rows
        out["best_under_0.7fp"] = max(ok, key=lambda r: r["mean_recall_5_50"]) if ok else None
        print("best:", out["best_under_0.7fp"])
    else:
        out["subsets"] = {}
        for s in a.subsets:
            out["subsets"][s] = run_subset(net, a.dataset_dir, s, a, device)
            print(s, out["subsets"][s], flush=True)
    if a.fps:
        out["fps"] = bench_fps(net, a, device)
        print("FPS:", out["fps"])
    if a.json:
        json.dump(out, open(a.json, "w"), indent=1)


if __name__ == "__main__":
    main()
'''); print('записано', '/kaggle/working/BasicIRSTD/eval_thermal.py')

In [ ]:
import os; os.makedirs(os.path.dirname('/kaggle/working/prepare_yolo.py'), exist_ok=True)
open('/kaggle/working/prepare_yolo.py', 'w').write(r'''"""Anti-UAV410 (+ відібрані негативи) -> YOLO-датасет і ті самі кадри для DNA-Net (маски з рамок).

  python prepare_yolo.py --root Anti-UAV-410 --neg negatives --out auav_yolo --stride 10 --neg_frac 0.15 --dnanet

Anti-UAV410: train/val/test/<серія>/*.jpg + IR_label.json {"exist": [...], "gt_rect": [[x, y, w, h], ...]}.
- Спліт офіційний, по серіях: серії не перетинаються між train, val і test.
- Сусідні кадри майже однакові, тому беремо кожен stride-й кадр із ціллю.
- Негативи: кадри без цілі з папки --neg/<split>, їх частка neg_frac від усіх кадрів спліту.
- DNA-Net вчиться на тих самих кадрах: маска = пікселі в рамці, що відрізняються від фону (Otsu), інакше еліпс.
"""
import os, glob, json, argparse, random
import numpy as np, cv2


def clip_box(box, W, H):
    """Рамку, що виходить за край кадру, обрізаємо по кадру. None, якщо від неї лишилось менше 1 px."""
    x, y, w, h = box
    x0, y0, x1, y1 = max(x, 0.0), max(y, 0.0), min(x + w, W), min(y + h, H)
    return (x0, y0, x1 - x0, y1 - y0) if x1 - x0 >= 1 and y1 - y0 >= 1 else None


def box_to_mask(img, boxes):
    H, W = img.shape
    m = np.zeros(img.shape, np.uint8)
    for b in boxes:
        b = clip_box(b, W, H)
        if b is None: continue
        x, y, w, h = b
        x0, y0 = int(x), int(y); x1, y1 = min(int(np.ceil(x + w)), W), min(int(np.ceil(y + h)), H)
        if x1 - x0 < 2 or y1 - y0 < 2: continue
        ell = np.zeros((y1 - y0, x1 - x0), np.uint8)
        cv2.ellipse(ell, ((x1 - x0) // 2, (y1 - y0) // 2), ((x1 - x0) // 2, (y1 - y0) // 2), 0, 0, 360, 1, -1)
        p = int(max(w, h) // 2 + 2)
        bg = np.median(img[max(y0 - p, 0):y1 + p, max(x0 - p, 0):x1 + p])
        diff = np.abs(img[y0:y1, x0:x1].astype(np.float32) - bg)
        _, th = cv2.threshold(cv2.normalize(diff, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8), 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        sel = th & ell
        if diff.max() < 4 or sel.sum() < 0.2 * ell.sum(): sel = ell
        m[y0:y1, x0:x1] |= sel * 255
    return m


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True); ap.add_argument("--neg"); ap.add_argument("--out", required=True)
    ap.add_argument("--stride", type=int, default=10); ap.add_argument("--neg_frac", type=float, default=0.15)
    ap.add_argument("--dnanet", action="store_true"); ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    rng = random.Random(a.seed); stats = {}
    for split in ["train", "val", "test"]:
        for d in ["images", "labels"] + (["masks"] if a.dnanet else []):
            os.makedirs(f"{a.out}/{d}/{split}", exist_ok=True)
        names, sizes = [], []
        for sd in sorted(glob.glob(f"{a.root}/{split}/*/")):
            seq = os.path.basename(sd.rstrip("/"))
            lf = next((f for f in ["IR_label.json", "infrared.json"] if os.path.exists(sd + f)), None)
            if lf is None: continue
            lab = json.load(open(sd + lf)); frames = sorted(glob.glob(sd + "*.jpg"))
            for i in range(0, min(len(frames), len(lab["exist"])), a.stride):
                box = lab["gt_rect"][i]
                if not lab["exist"][i] or len(box) != 4 or box[2] < 1 or box[3] < 1: continue
                img = cv2.imread(frames[i], 0); H, W = img.shape
                box = clip_box(box, W, H)                       # частина рамок Anti-UAV410 виходить за край кадру
                if box is None: continue
                x, y, w, h = box; n = f"{seq}_{i:06d}"
                cv2.imwrite(f"{a.out}/images/{split}/{n}.jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 95])
                open(f"{a.out}/labels/{split}/{n}.txt", "w").write(f"0 {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}\n")
                if a.dnanet: cv2.imwrite(f"{a.out}/masks/{split}/{n}.png", box_to_mask(img, [box]))
                names.append(n); sizes.append(max(w, h))
        n_pos = len(names)
        negs = sorted(glob.glob(f"{a.neg}/{split}/*.jpg")) if a.neg else []
        k = min(len(negs), round(a.neg_frac / (1 - a.neg_frac) * n_pos)); rng.shuffle(negs)
        for p in negs[:k]:
            n = "neg_" + os.path.basename(p)[:-4]; img = cv2.imread(p, 0)
            cv2.imwrite(f"{a.out}/images/{split}/{n}.jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 95])
            open(f"{a.out}/labels/{split}/{n}.txt", "w").close()
            if a.dnanet: cv2.imwrite(f"{a.out}/masks/{split}/{n}.png", np.zeros_like(img))
            names.append(n)
        s = np.array(sizes)
        stats[split] = dict(positives=n_pos, negatives=k, neg_share=round(k / max(len(names), 1), 3),
                            box_lt5=round(float((s < 5).mean()), 3) if len(s) else None,
                            box_5_50=round(float(((s >= 5) & (s <= 50)).mean()), 3) if len(s) else None,
                            box_gt50=round(float((s > 50).mean()), 3) if len(s) else None)
        if a.dnanet:
            os.makedirs(f"{a.out}/img_idx", exist_ok=True)
            open(f"{a.out}/img_idx/{split}.txt", "w").write("\n".join(f"{split}/{n}" for n in names))
        print(split, stats[split], flush=True)
    open(f"{a.out}/data.yaml", "w").write(f"path: {os.path.abspath(a.out)}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: target\n")
    json.dump(stats, open(f"{a.out}/stats.json", "w"), indent=1)


if __name__ == "__main__":
    main()
'''); print('записано', '/kaggle/working/prepare_yolo.py')

In [ ]:
# ======= ПАКЕТ КОМАНДИ AntiUAV410_SKY: builder і маніфест відібраних кадрів (вже проріджений під цей ноутбук) =======
import os, base64
os.makedirs('/tmp/dl/team_package', exist_ok=True)
open('/tmp/dl/team_package/build_antiuav_yolo.py', 'w').write(r'''#!/usr/bin/env python3
"""Build a single-class 640x512 YOLO dataset from a curated manifest and RAW ZIP.

Only selected image bytes are read. Inputs are read-only. Nonempty output directories
are refused. Exit codes: 0 complete; 1 partial build with row errors; 2 fatal input error.
"""
from __future__ import annotations
import argparse
import csv
import hashlib
import io
import json
import math
import re
import sys
import zipfile
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath

import pandas as pd
from PIL import Image
import yaml
from tqdm import tqdm

VERSION = '1.1.0'
SPLITS = ('train','val','test')
ROLES = ('positive','negative','hard_negative')
REQUIRED = {'sequence','frame_number','image_name','exist','valid_bbox','bbox_x','bbox_y','bbox_w','bbox_h','sample_role','assigned_split'}
ERROR_FIELDS = ['row','sequence','frame_number','image_name','error','detail']

def digest(path: Path) -> str:
    h=hashlib.sha256()
    with path.open('rb') as src:
        for block in iter(lambda:src.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

def boolean(value) -> bool:
    text=str(value).strip().lower()
    if text in ('true','1','1.0'):return True
    if text in ('false','0','0.0'):return False
    raise ValueError(f'Invalid boolean {value!r}')

def output_stem(sequence: str, frame: int) -> str:
    safe=re.sub(r'[^A-Za-z0-9_-]','_',sequence)[:80]
    key=hashlib.sha256(sequence.encode('utf-8')).hexdigest()[:16]
    return f'{safe}__{key}__{frame:08d}'

def load_manifest(path: Path) -> pd.DataFrame:
    df=pd.read_csv(path,dtype={'sequence':str,'image_name':str,'sample_role':str,'assigned_split':str})
    missing=REQUIRED-set(df.columns)
    if missing:raise ValueError(f'Manifest missing columns: {sorted(missing)}')
    if df.empty:raise ValueError('Manifest is empty')
    for col in ['sequence','image_name','sample_role','assigned_split']:
        if df[col].isna().any() or df[col].str.strip().eq('').any():raise ValueError(f'Blank {col}')
    if not set(df.assigned_split)<=set(SPLITS):raise ValueError('Only train/val/test splits are allowed')
    if not set(df.sample_role)<=set(ROLES):raise ValueError('Unknown sample_role')
    for col in ['frame_number','exist']:
        vals=pd.to_numeric(df[col],errors='raise')
        if vals.isna().any() or not vals.map(lambda x:math.isfinite(x) and x==int(x)).all():raise ValueError(f'Invalid integer {col}')
        df[col]=vals.astype('int64')
    if (df.frame_number<1).any() or not set(df.exist)<={0,1}:raise ValueError('Invalid frame_number or exist')
    if df.duplicated(['sequence','frame_number']).any() or df.duplicated(['sequence','image_name']).any():raise ValueError('Duplicate frame keys')
    if (df.groupby('sequence').assigned_split.nunique()>1).any():raise ValueError('Sequence appears in multiple splits')
    if 'recording_group' in df:
        if df.recording_group.isna().any() or df.recording_group.astype(str).str.strip().eq('').any():raise ValueError('Blank recording_group')
        if (df.groupby('recording_group').assigned_split.nunique()>1).any():raise ValueError('Recording group appears in multiple splits')
    for row in df.itertuples():
        if '/' in row.sequence or '\\' in row.sequence or row.sequence in ('.','..'):raise ValueError('Sequence must be a directory component')
        if PurePosixPath(row.image_name).name!=row.image_name or '\\' in row.image_name or not row.image_name.lower().endswith(('.jpg','.jpeg')):raise ValueError('image_name must be a JPEG basename')
    df['valid_bbox']=df.valid_bbox.map(boolean)
    return df.sort_values(['assigned_split','sequence','frame_number']).reset_index(drop=True)

def index_zip(z: zipfile.ZipFile, sequences: set[str]):
    index=defaultdict(list)
    # ZIP central-directory metadata only. No unselected image is opened/extracted.
    for info in tqdm(z.infolist(),desc='Index ZIP',unit='entries'):
        if info.is_dir():continue
        parts=PurePosixPath(info.filename.replace('\\','/')).parts
        if not parts or not parts[-1].lower().endswith(('.jpg','.jpeg')):continue
        hits=sequences.intersection(parts[:-1])
        for seq in hits:index[(seq,parts[-1].lower())].append((info,parts))
    return index

def resolve_member(index, row):
    candidates=index.get((row.sequence,row.image_name.lower()),[])
    if not candidates:raise FileNotFoundError('Selected JPG not found under sequence directory')
    thermal=[c for c in candidates if any(p.lower() in {'infrared','ir','thermal'} for p in c[1][:-1])]
    if thermal:candidates=thermal
    else:
        candidates=[c for c in candidates if not any(p.lower() in {'visible','rgb','color'} for p in c[1][:-1])]
    if len(candidates)>1 and hasattr(row,'original_split'):
        same=[c for c in candidates if str(row.original_split).lower() in [p.lower() for p in c[1][:-1]]]
        if same:candidates=same
    if len(candidates)!=1:raise ValueError(f'Ambiguous or non-thermal JPG match: {len(candidates)} candidates')
    return candidates[0][0]

def make_label(row, width: int, height: int) -> str:
    if row.sample_role in ('negative','hard_negative'):
        if row.exist!=0 or row.valid_bbox:raise ValueError('Negative role contradicts exist/valid_bbox')
        return ''
    if row.exist!=1 or not row.valid_bbox:raise ValueError('Invalid positive bbox or contradictory exist')
    x,y,w,h=(float(getattr(row,f'bbox_{k}')) for k in ('x','y','w','h'))
    if not all(math.isfinite(v) for v in (x,y,w,h)):raise ValueError('Nonfinite bbox')
    if w<=0 or h<=0 or x<0 or y<0 or x+w>width or y+h>height:raise ValueError(f'BBox outside {width}x{height}: {(x,y,w,h)}')
    norm=((x+w/2)/width,(y+h/2)/height,w/width,h/height)
    if not all(0<=v<=1 for v in norm):raise ValueError('Normalized bbox outside [0,1]')
    return '0 '+' '.join(f'{v:.10f}' for v in norm)+'\n'

def preflight(source, manifest, df, out) -> int:
    """Validate every selected reference using ZIP metadata; no JPEG payloads read."""
    errors=[]; seen=set(); stems=set(); image_bytes=0
    try:
        with zipfile.ZipFile(source,'r') as z:
            index=index_zip(z,set(df.sequence))
            for number,row in enumerate(tqdm(df.itertuples(index=False),total=len(df),desc='Check references',unit='frames'),start=2):
                try:
                    make_label(row,640,512)
                    info=resolve_member(index,row)
                    if info.header_offset in seen:raise ValueError('Same RAW member referenced twice')
                    stem=output_stem(row.sequence,row.frame_number)
                    if stem in stems:raise ValueError('Output name collision')
                    if info.flag_bits & 1:raise ValueError('Encrypted selected ZIP member')
                    if info.file_size==0:raise ValueError('Empty selected JPEG member')
                    if info.compress_type not in {zipfile.ZIP_STORED,zipfile.ZIP_DEFLATED,zipfile.ZIP_BZIP2,zipfile.ZIP_LZMA}:raise ValueError('Unsupported ZIP compression')
                    seen.add(info.header_offset);stems.add(stem);image_bytes+=info.file_size
                except (ValueError,OSError) as exc:
                    errors.append({'row':number,'sequence':row.sequence,'frame_number':row.frame_number,'image_name':row.image_name,'error':type(exc).__name__,'detail':str(exc)})
    except (OSError,zipfile.BadZipFile,RuntimeError) as exc:
        errors.append({'row':'','sequence':'','frame_number':'','image_name':'','error':type(exc).__name__,'detail':'Fatal archive error: '+str(exc)})
    with (out/'preflight_errors.csv').open('w',encoding='utf-8',newline='') as dest:
        writer=csv.DictWriter(dest,fieldnames=ERROR_FIELDS);writer.writeheader();writer.writerows(errors)
    report={'builder_version':VERSION,'status':'metadata_pass' if not errors else 'failed','source':str(source),'source_bytes':source.stat().st_size,'manifest_sha256':digest(manifest),'expected_frames':len(df),'matched_frames':len(seen),'errors':len(errors),'selected_image_bytes':image_bytes,'selected_image_gib':round(image_bytes/(1024**3),3),'expected_output_files':2*len(df)+3,'by_split':df.assigned_split.value_counts().to_dict(),'by_role':df.sample_role.value_counts().to_dict(),'limitations':['ZIP metadata only: JPEG dimensions, decoding, CRC and EXIF are checked during full build.','Image bytes exclude labels, filesystem allocation overhead and report files. Allow additional free space.','No annotation-completeness or cross-sequence image-duplicate audit performed.']}
    (out/'preflight_report.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
    print(f"{report['status'].upper()}: {len(seen)}/{len(df)} references, {len(errors)} errors; images {report['selected_image_gib']} GiB. Report: {out/'preflight_report.json'}")
    return 0 if not errors else 1

def run(args) -> int:
    source=args.source.resolve();manifest=args.manifest.resolve();out=args.output.resolve()
    if not source.is_file() or not manifest.is_file():raise ValueError('Source ZIP and manifest must be existing files')
    if out.exists() and (not out.is_dir() or any(out.iterdir())):raise ValueError('Output must be absent or an empty directory; refusing overwrite')
    df=load_manifest(manifest)
    out.mkdir(parents=True,exist_ok=True)
    if getattr(args,'check_only',False):return preflight(source,manifest,df,out)
    report={'builder_version':VERSION,'source':str(source),'source_bytes':source.stat().st_size,'source_sha256':digest(source) if args.hash_source else None,'manifest':str(manifest),'manifest_sha256':digest(manifest),'expected_frames':len(df),'dimensions':[640,512],'class_names':{'0':'thermal_target'},'status':'building','exported_frames':0,'errors':0,'by_split':{},'by_role':{},'notes':['Strict 640x512; no resizing, clipping, or silent annotation repair.','Positive boxes invalid or out of bounds are skipped, never converted to negative.','Thermal directory preferred; ambiguous matches skipped.','Only selected JPEG payloads read; ZIP central directory indexed.']}
    errors=[];counts=Counter();roles=Counter();seen_members=set();seen_output=set()
    for split in SPLITS:
        for category in ('images','labels'):(out/category/split).mkdir(parents=True,exist_ok=True)
    try:
        with zipfile.ZipFile(source,'r') as z:
            index=index_zip(z,set(df.sequence))
            for number,row in enumerate(tqdm(df.itertuples(index=False),total=len(df),desc='Build YOLO',unit='frames'),start=2):
                image_path=label_path=None
                try:
                    # Validate label semantics before opening an image, including invalid flags.
                    label=make_label(row,640,512)
                    info=resolve_member(index,row)
                    if info.header_offset in seen_members:raise ValueError('Same RAW member referenced twice')
                    raw=z.read(info)
                    with Image.open(io.BytesIO(raw)) as im:
                        if im.format!='JPEG':raise ValueError('Payload is not JPEG')
                        if im.size!=(640,512):raise ValueError(f'Expected 640x512, got {im.size}')
                        if im.mode not in ('L','RGB'):raise ValueError(f'Unsupported image mode {im.mode}')
                        if im.getexif().get(274,1)!=1:raise ValueError('Nontrivial EXIF orientation; coordinate interpretation requires review')
                        im.load() # decode completely; corrupt/truncated JPEG is an error
                    stem=output_stem(row.sequence,row.frame_number)
                    if stem in seen_output:raise ValueError('Output name collision')
                    image_path=out/'images'/row.assigned_split/(stem+'.jpg')
                    label_path=out/'labels'/row.assigned_split/(stem+'.txt')
                    with image_path.open('xb') as dest:dest.write(raw)
                    with label_path.open('x',encoding='utf-8',newline='\n') as dest:dest.write(label)
                    seen_output.add(stem);seen_members.add(info.header_offset)
                    counts[row.assigned_split]+=1;roles[row.sample_role]+=1
                except (ValueError,OSError,zipfile.BadZipFile,RuntimeError,EOFError) as exc:
                    # Only paths created by this row, under an initially empty output, are cleaned.
                    for target in (image_path,label_path):
                        if target is not None and target.is_file():target.unlink()
                    errors.append({'row':number,'sequence':row.sequence,'frame_number':row.frame_number,'image_name':row.image_name,'error':type(exc).__name__,'detail':str(exc)})
    except (OSError,zipfile.BadZipFile,RuntimeError) as exc:
        errors.append({'row':'','sequence':'','frame_number':'','image_name':'','error':type(exc).__name__,'detail':'Fatal archive error: '+str(exc)})
    with (out/'build_errors.csv').open('w',encoding='utf-8',newline='') as dest:
        writer=csv.DictWriter(dest,fieldnames=ERROR_FIELDS);writer.writeheader();writer.writerows(errors)
    yaml_data={'path':out.as_posix(),'train':'images/train','val':'images/val','test':'images/test','names':{0:'thermal_target'}}
    (out/'data.yaml').write_text(yaml.safe_dump(yaml_data,sort_keys=False,allow_unicode=True),encoding='utf-8')
    report.update(exported_frames=sum(counts.values()),errors=len(errors),by_split={s:counts[s] for s in SPLITS},by_role={r:roles[r] for r in ROLES},status='complete' if not errors and sum(counts.values())==len(df) else 'partial')
    report['file_counts']={s:{'images':len(list((out/'images'/s).glob('*.jpg'))),'labels':len(list((out/'labels'/s).glob('*.txt')))} for s in SPLITS}
    (out/'build_report.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
    print(f"{report['status'].upper()}: {report['exported_frames']}/{len(df)} frames, {len(errors)} errors. Report: {out/'build_report.json'}")
    return 0 if report['status']=='complete' else 1

def main() -> int:
    p=argparse.ArgumentParser(description=__doc__,formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument('--source',required=True,type=Path,help='Read-only AntiUAV410_RAW.zip')
    p.add_argument('--manifest',required=True,type=Path,help='selected_frames.csv or .csv.gz')
    p.add_argument('--output',required=True,type=Path,help='New or empty output directory')
    p.add_argument('--hash-source',action='store_true',help='Optionally stream entire ZIP to compute SHA-256; adds a full-file read')
    p.add_argument('--check-only',action='store_true',help='Check all selected ZIP paths and label semantics without reading JPEG payloads; write preflight reports only. Use a separate empty output directory.')
    args=p.parse_args()
    if args.check_only and args.hash_source:p.error('--hash-source cannot be combined with metadata-only --check-only')
    try:return run(args)
    except (ValueError,OSError,KeyError) as exc:
        print(f'FATAL: {exc}',file=sys.stderr);return 2
if __name__=='__main__':raise SystemExit(main())
''')
open('/tmp/dl/team_package/selected_frames.csv.gz', 'wb').write(base64.b64decode('...'))
print('пакет команди записано')

In [ ]:
%cd /kaggle/working
# ======= ДАНІ: командний датасет Anti-UAV410 SKY + синтетичне сонце =======
DRIVE_URL = 'https://drive.google.com/file/d/1zsdazmKS3mHaEZWS2BnqbYHPEcIaH5WR/view'   # AntiUAV410.zip (сирий архів)
PKG_URL = ''   # не потрібен: пакет команди (маніфест відібраних кадрів + builder) вбудовано в клітинку вище
TRAIN_EVERY = 1   # маніфест у клітинці вище вже проріджений (train: кожен 6-й кадр із ціллю, val: кожен 4-й), тут більше не проріджуємо
VAL_EVERY = 1     # val у маніфесті вже кожен 4-й кадр. Test тут не збираємо: фінальний тест робимо на 2080 SUPER з повним пакетом
SPLITS = ['train', 'val']
SUN_FRAC = 0.10                 # до train додаємо копії 10% кадрів із синтетичним сонцем (ціль лишається, сонце за 40+ px від неї)

import os, glob, re, subprocess, zipfile, shutil, random, yaml
import numpy as np, pandas as pd, cv2
DL = '/tmp/dl'; os.makedirs(DL, exist_ok=True)

def fetch(url, out):
    # Великі файли Google Drive віддає через сторінку "не вдалося перевірити на віруси".
    # Качаємо напряму з drive.usercontent.google.com з confirm=t, а якщо прийшла сторінка, беремо з неї форму і пробуємо ще раз.
    import requests
    if os.path.exists(out) and os.path.getsize(out) > 1 << 20: print('вже є', out); return
    fid = (re.search(r'/d/([\w-]{20,})', url) or re.search(r'id=([\w-]{20,})', url)).group(1)
    s, u, params = requests.Session(), 'https://drive.usercontent.google.com/download', {'id': fid, 'export': 'download', 'confirm': 't'}
    for _ in range(3):
        r = s.get(u, params=params, stream=True, timeout=120)
        if 'text/html' not in r.headers.get('Content-Type', ''):
            total, done, mark = int(r.headers.get('Content-Length', 0)), 0, 0
            with open(out, 'wb') as f:
                for chunk in r.iter_content(8 << 20):
                    f.write(chunk); done += len(chunk)
                    if done - mark > 1 << 30: mark = done; print(f'  {done / 2**30:.1f} / {total / 2**30:.1f} ГБ', flush=True)
            print(f'скачано {out}: {done / 2**30:.2f} ГБ'); return
        html = r.text
        form = re.search(r'<form[^>]+action="([^"]+)"', html)
        inputs = dict(re.findall(r'<input[^>]*name="([^"]+)"[^>]*value="([^"]*)"', html))
        if not (form and inputs): break
        u, params = form.group(1).replace('&amp;', '&'), inputs
    t = re.findall(r'<title>(.*?)</title>', html, re.S)
    raise SystemExit('Google Drive не віддав файл: ' + (t[0].strip() if t else '?') + '. Перевірте доступ "Усі, хто має посилання".')

# 1. Сирий архів і пакет команди
RAW = f'{DL}/AntiUAV410.zip'; fetch(DRIVE_URL, RAW)
assert zipfile.is_zipfile(RAW), 'DRIVE_URL має вести на zip-архів Anti-UAV410'
if not os.path.exists(f'{DL}/team_package/build_antiuav_yolo.py') and PKG_URL.startswith('http'):
    fetch(PKG_URL, f'{DL}/pkg.zip'); zipfile.ZipFile(f'{DL}/pkg.zip').extractall(DL)
PKG = os.path.dirname(glob.glob(f'{DL}/**/build_antiuav_yolo.py', recursive=True)[0])
subprocess.run('pip install -q "pandas>=2.2" "Pillow>=10" "PyYAML>=6" tqdm', shell=True)

# 2. Проріджуємо маніфест: train кожен TRAIN_EVERY-й позитив у серії, val кожен VAL_EVERY-й, негативи і test усі
m = pd.read_csv(f'{PKG}/selected_frames.csv.gz', dtype={'sequence': str, 'image_name': str, 'sample_role': str, 'assigned_split': str})
m = m[m.assigned_split.isin(SPLITS)].sort_values(['assigned_split', 'sequence', 'frame_number'])
pos = m.sample_role == 'positive'
rank = m[pos].groupby(['assigned_split', 'sequence']).cumcount()
step = m.loc[pos, 'assigned_split'].map({'train': TRAIN_EVERY, 'val': VAL_EVERY, 'test': 1})
keep = pd.Series(True, index=m.index); keep[pos] = (rank % step == 0)
m = m[keep]
MAN = f'{DL}/manifest_used.csv'; m.to_csv(MAN, index=False)
print('кадрів у збірці:', m.groupby(['assigned_split', 'sample_role']).size().to_dict())

# 3. Збірка командним builder: перевірка шляхів, потім повна збірка (кожен JPEG декодується і перевіряється)
SRC = '/tmp/sky_yolo'
if not os.path.exists(f'{SRC}/data.yaml'):
    shutil.rmtree('/tmp/sky_check', ignore_errors=True); shutil.rmtree(SRC, ignore_errors=True)
    r = subprocess.run(f'python {PKG}/build_antiuav_yolo.py --source {RAW} --manifest {MAN} --output /tmp/sky_check --check-only', shell=True)
    if r.returncode != 0:
        if os.path.exists('/tmp/sky_check/preflight_errors.csv'): print(open('/tmp/sky_check/preflight_errors.csv').read()[:3000])
        raise SystemExit('Перевірка архіву не пройшла: шляхи в архіві не збігаються з маніфестом. Можливо, на Drive інший архів, ніж у того, хто робив пакет.')
    r = subprocess.run(f'python {PKG}/build_antiuav_yolo.py --source {RAW} --manifest {MAN} --output {SRC}', shell=True)
    errs = pd.read_csv(f'{SRC}/build_errors.csv')
    print('помилок збірки:', len(errs))
    if len(errs) > 0.01 * len(m): raise SystemExit('Забагато помилок збірки, див. /tmp/sky_yolo/build_errors.csv')

# 4. Синтетичне сонце: копії частини кадрів train із пересвіченим диском і ореолом
def add_sun(img, boxes, rng):
    h, w = img.shape
    r = rng.uniform(8, 30); sig = rng.uniform(1.0, 3.0) * r
    for _ in range(200):
        if boxes and rng.random() < 0.5:                       # половина сонць поруч із ціллю (40-120 px)
            bx, by = boxes[rng.integers(len(boxes))]; ang = rng.uniform(0, 2 * np.pi); d = rng.uniform(40, 120) + r
            cx, cy = bx + d * np.cos(ang), by + d * np.sin(ang)
        else:
            cx, cy = rng.uniform(r, w - r), rng.uniform(r, h * 0.6)
        if not (r <= cx < w - r and r <= cy < h - r): continue
        if boxes and min(np.hypot(bx - cx, by - cy) for bx, by in boxes) < 40 + r: continue
        break
    yy, xx = np.mgrid[:h, :w]; d = np.hypot(xx - cx, yy - cy)
    halo = np.where(d <= r, 1.0, np.exp(-((d - r) ** 2) / (2 * sig ** 2)))
    out = img.astype(np.float32) + (255 - img.astype(np.float32)) * halo * rng.uniform(0.6, 1.0)
    out[d <= r] = 255
    return np.clip(out, 0, 255).astype(np.uint8)

if SUN_FRAC > 0 and not glob.glob(f'{SRC}/images/train/sun_*'):
    rng = np.random.default_rng(0)
    files = sorted(glob.glob(f'{SRC}/images/train/*.jpg')); random.Random(0).shuffle(files)
    for p in files[:int(len(files) * SUN_FRAC)]:
        n = os.path.basename(p)[:-4]; img = cv2.imread(p, 0); H, W = img.shape
        lab = open(f'{SRC}/labels/train/{n}.txt').read()
        centers = [(float(l.split()[1]) * W, float(l.split()[2]) * H) for l in lab.splitlines() if l.strip()]
        cv2.imwrite(f'{SRC}/images/train/sun_{n}.jpg', add_sun(img, centers, rng), [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{SRC}/labels/train/sun_{n}.txt', 'w').write(lab)

d = yaml.safe_load(open(f'{SRC}/data.yaml')); d['path'] = SRC
yaml.safe_dump(d, open('/kaggle/working/data.yaml', 'w'))
for s in SPLITS:
    labs = glob.glob(f'{SRC}/labels/{s}/*.txt')
    neg = sum(os.path.getsize(f) == 0 for f in labs); sun = len(glob.glob(f'{SRC}/images/{s}/sun_*'))
    print(f'{s}: {len(labs)} кадрів, негативів {neg} ({neg / max(len(labs), 1):.1%}), із сонцем {sun}')


In [ ]:
%cd /kaggle/working/BasicIRSTD
!python patch_basicirstd.py

## Датасет у форматі BasicIRSTD
`datasets/AUAV/images/*.png`, `masks/*.png`, `img_idx/train_AUAV.txt`, `test_AUAV.txt` (= val).

In [ ]:
import sys, cv2, numpy as np
sys.path.insert(0, '/kaggle/working')
from prepare_yolo import box_to_mask
OUT = '/tmp/dnanet_ds/AUAV'   # не в /kaggle/working, щоб тисячі png не йшли в Output
os.makedirs(f'{OUT}/images', exist_ok=True); os.makedirs(f'{OUT}/masks', exist_ok=True); os.makedirs(f'{OUT}/img_idx', exist_ok=True)
lists = {}
for split in ['train', 'val']:
    files = sorted(glob.glob(f'{SRC}/images/{split}/*'))
    if split == 'val': files = files[::VAL_EVERY]
    names = []
    for p in files:
        n = os.path.basename(p).rsplit('.', 1)[0]; img = cv2.imread(p, 0); H, W = img.shape
        boxes = []
        lf = f'{SRC}/labels/{split}/{n}.txt'
        if os.path.exists(lf):
            for l in open(lf).read().split('\n'):
                if l.strip():
                    _, cx, cy, w, h = map(float, l.split()); boxes.append((cx * W - w * W / 2, cy * H - h * H / 2, w * W, h * H))
        cv2.imwrite(f'{OUT}/images/{n}.png', img); cv2.imwrite(f'{OUT}/masks/{n}.png', box_to_mask(img, boxes)); names.append(n)
    lists[split] = names
open(f'{OUT}/img_idx/train_AUAV.txt', 'w').write('\n'.join(lists['train']))
open(f'{OUT}/img_idx/test_AUAV.txt', 'w').write('\n'.join(lists['val']))
sample = np.stack([cv2.imread(f'{OUT}/images/{n}.png', 0) for n in lists['train'][::max(1, len(lists['train']) // 500)]]).astype(np.float32)
MEAN, STD = float(sample.mean()), float(sample.std()); print('train', len(lists['train']), 'val', len(lists['val']), 'mean', MEAN, 'std', STD)

## Навчання
Чекпойнт кожні 5 епох у `/kaggle/working/log`. Якщо час вийде раніше, лишиться останній збережений.

In [ ]:
import subprocess
cmd = (f'CUDA_VISIBLE_DEVICES=0 python train.py --model_names DNANet --dataset_names AUAV --batchSize {BATCH} --patchSize {PATCH} --nEpochs {EPOCHS} '
       f'--dataset_dir /tmp/dnanet_ds --save /kaggle/working/log --threads 4 --save_every 5 --img_norm_cfg_mean {MEAN} --img_norm_cfg_std {STD}')
import os, signal
p = subprocess.Popen(cmd, shell=True, start_new_session=True)   # окрема група процесів, щоб при таймауті вбити і сам train.py
try:
    p.wait(timeout=HOURS * 3600)
except subprocess.TimeoutExpired:
    os.killpg(p.pid, signal.SIGKILL); p.wait()
    print('час вийшов, навчання зупинено, беремо останній чекпойнт')

## Оцінка за ТЗ на val
Ті самі правила, що й для YOLO: Recall цілей 5-50 px, FP на кадр, поріг з перебору.

In [ ]:
import re
ck = sorted(glob.glob('/kaggle/working/log/AUAV/DNANet_*.pth.tar'), key=lambda p: int(re.findall(r'_(\d+)\.pth', p)[0]))[-1]; print(ck)
!CUDA_VISIBLE_DEVICES=0 python eval_thermal.py --model DNANet --ckpt {ck} --dataset_dir /tmp/dnanet_ds --subsets AUAV --mean {MEAN} --std {STD} --sun --sweep --json /kaggle/working/metrics_sweep.json | tail -20
!CUDA_VISIBLE_DEVICES=0 python eval_thermal.py --model DNANet --ckpt {ck} --dataset_dir /tmp/dnanet_ds --subsets AUAV --mean {MEAN} --std {STD} --sun --topk 3 --fps --json /kaggle/working/metrics.json | tail -20
json.dump(dict(mean=MEAN, std=STD, ckpt=ck), open('/kaggle/working/norm.json', 'w'))